# Tuned DGM Model Analysis\n\nThis notebook compares observed DGM (`DGM_Valid_CAO.csv`) with model outputs from `Unit_WorM3_CAO_All_Calib_Direct.py`.\nIt uses optimized `ko_UVA` and `RHg_UVA` coefficients and reports baseline vs tuned statistics and plots.\n

In [ ]:
import numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\nfrom Unit_WorM3_CAO_All_Calib_Direct import run_model\n\nBEST_COEFFS = {\n    'ko_uva_num': 0.350407009078024,\n    'rhg_uva_a': -0.02243776483632441,\n    'rhg_uva_b': 1.4911911655729884,\n    'rhg_uva_c': -23.612521190306616,\n}\n\nprint('Best coefficients:', BEST_COEFFS)\n

In [ ]:
# Baseline model outputs (already generated by default run)\nbaseline_df = pd.read_csv('Hg_budget_summary.csv')\nbaseline_metrics = pd.read_csv('Hg_model_metrics.csv')\n\n# Tuned model outputs (recompute to ensure consistency with BEST_COEFFS)\ntuned_df, tuned_metrics = run_model(\n    input_csv='DGM_Valid_CAO.csv',\n    output_summary_csv='Hg_budget_summary_tuned_best.csv',\n    output_metrics_csv='Hg_model_metrics_tuned_best.csv',\n    tuning=BEST_COEFFS,\n    verbose=False,\n)\n\ndisplay(baseline_df.head(3))\ndisplay(tuned_df.head(3))\n

In [ ]:
def summarize_fit(df: pd.DataFrame):\n    obs = df['DGM_obs'].to_numpy(dtype=float)\n    sim = df['Hg0_water'].to_numpy(dtype=float)\n    sal = df['Salinity'].to_numpy(dtype=float)\n    valid = np.isfinite(obs) & np.isfinite(sim) & np.isfinite(sal)\n    obs = obs[valid]\n    sim = sim[valid]\n    sal = sal[valid]\n\n    p = np.polyfit(obs, sim, 1)\n    slope = float(p[0])\n    intercept = float(p[1])\n    sim_fit = slope * obs + intercept\n\n    ss_res_fit = float(np.sum((sim - sim_fit) ** 2))\n    ss_tot_fit = float(np.sum((sim - np.mean(sim)) ** 2))\n    r2_reg = 1 - ss_res_fit / ss_tot_fit\n\n    obs_mean = float(np.mean(obs))\n    sim_mean = float(np.mean(sim))\n    num = float(np.sum((obs - obs_mean) * (sim - sim_mean)))\n    den = float(np.sqrt(np.sum((obs - obs_mean) ** 2) * np.sum((sim - sim_mean) ** 2)))\n    r = num / den\n    r2_corr = r ** 2\n\n    rmse = float(np.sqrt(np.mean((sim - obs) ** 2)))\n    pbias = float(100 * np.sum(obs - sim) / np.sum(obs))\n    mean_bias = float(np.mean(sim - obs))\n\n    slope_obs_sal = float(np.polyfit(sal, obs, 1)[0])\n    slope_sim_sal = float(np.polyfit(sal, sim, 1)[0])\n\n    return {\n        'N': int(obs.size),\n        'R2_reg': r2_reg,\n        'R2_corr': r2_corr,\n        'r': r,\n        'RMSE': rmse,\n        'Slope': slope,\n        'Intercept': intercept,\n        'PBIAS_percent': pbias,\n        'MeanBias': mean_bias,\n        'SalinitySlope_obs': slope_obs_sal,\n        'SalinitySlope_model': slope_sim_sal,\n        'SalinitySlope_gap': slope_sim_sal - slope_obs_sal,\n    }\n\nbase_stats = summarize_fit(baseline_df)\ntuned_stats = summarize_fit(tuned_df)\n\nstats_df = pd.DataFrame([base_stats, tuned_stats], index=['baseline', 'tuned'])\ndisplay(stats_df)\ndisplay((stats_df.loc['tuned'] - stats_df.loc['baseline']).to_frame('tuned_minus_baseline').T)\n

In [ ]:
# Variable-wise comparison: observed DGM vs tuned model DGM\nfig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)\nspecs = [\n    ('Salinity', 'Salinity (psu)'),\n    ('Temperature', 'Temperature (°C)'),\n    ('SeaIce', 'Sea ice fraction'),\n    ('WindSpeed', 'Wind speed (m/s)'),\n]\n\nfor ax, (col, label) in zip(axes.ravel(), specs):\n    x = tuned_df[col].to_numpy(dtype=float)\n    y_obs = tuned_df['DGM_obs'].to_numpy(dtype=float)\n    y_sim = tuned_df['Hg0_water'].to_numpy(dtype=float)\n\n    ax.scatter(x, y_obs, s=24, alpha=0.75, label='Observed DGM', color='black')\n    ax.scatter(x, y_sim, s=24, alpha=0.75, label='Tuned model DGM', color='tab:blue')\n    ax.set_xlabel(label)\n    ax.set_ylabel('DGM (ng/L)')\n    ax.grid(True, alpha=0.25)\n    ax.legend(loc='best', fontsize=8)\n\nfig.suptitle('Observed vs Tuned Model DGM by Input Variable')\nfig.savefig('fig_tuned_variable_comparison.png', dpi=220)\nplt.show()\n

In [ ]:
# Observed vs model DGM relationship (baseline and tuned)\nfig, ax = plt.subplots(figsize=(7.5, 6.5))\n\nobs_base = baseline_df['DGM_obs'].to_numpy(dtype=float)\nsim_base = baseline_df['Hg0_water'].to_numpy(dtype=float)\nobs_tune = tuned_df['DGM_obs'].to_numpy(dtype=float)\nsim_tune = tuned_df['Hg0_water'].to_numpy(dtype=float)\n\nax.scatter(obs_base, sim_base, s=28, alpha=0.55, label='Baseline', color='tab:orange')\nax.scatter(obs_tune, sim_tune, s=28, alpha=0.75, label='Tuned', color='tab:blue')\n\nmn = float(min(obs_tune.min(), sim_tune.min(), obs_base.min(), sim_base.min()))\nmx = float(max(obs_tune.max(), sim_tune.max(), obs_base.max(), sim_base.max()))\nax.plot([mn, mx], [mn, mx], 'k--', lw=1.2, label='1:1 line')\n\nm_b, b_b = np.polyfit(obs_base, sim_base, 1)\nm_t, b_t = np.polyfit(obs_tune, sim_tune, 1)\nxline = np.linspace(mn, mx, 200)\nax.plot(xline, m_b * xline + b_b, color='tab:orange', lw=1.6, label=f'Baseline fit: y={m_b:.3f}x+{b_b:.3f}')\nax.plot(xline, m_t * xline + b_t, color='tab:blue', lw=1.8, label=f'Tuned fit: y={m_t:.3f}x+{b_t:.3f}')\n\nax.set_xlabel('Observed DGM (ng/L)')\nax.set_ylabel('Modeled DGM (ng/L)')\nax.set_title('Observed vs Modeled DGM: Baseline vs Tuned')\nax.grid(True, alpha=0.25)\nax.legend(fontsize=8)\nfig.savefig('fig_obs_vs_model_baseline_vs_tuned.png', dpi=220)\nplt.show()\n